# Обученная модель: слабый сигнал, зрелая технология или шум

Нужны две библиотеки: **pandas** для таблиц и **scikit-learn** для модели.
`pickle` входит в Python и просто сохраняет результат обучения.
Открывайте ноутбук из папки комплекта, чтобы он видел соседнюю папку `Данные`.

**Что мы определяем:** о чём конкретная публикация — о ранней технологии,
о уже используемой технологии или о событии без нужного технологического сигнала.

**Граница результата:** проверена классификация английских публикаций и общий
топ-15 на подготовленной выборке.

## 1. Подключаем инструменты

In [ ]:
import pickle
import pandas as pd
from sklearn.pipeline import make_pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

## 2. Читаем подготовленные примеры

In [ ]:
данные = pd.read_csv("Данные/Обучение.csv", keep_default_na=False)
print("Всего публикаций:", len(данные))
pd.crosstab(данные["часть"], данные["класс"])

Всего публикаций: 248


класс,зрелая технология,слабый сигнал,шум
часть,,,
обучение,21,113,38
проверка,8,51,17


## 3. Разделяем обучение и проверку

На 172 публикациях модель учится. 76 других публикаций оставлены для проверки.

In [ ]:
обучение = данные[данные["часть"] == "обучение"].copy()
проверка = данные[данные["часть"] == "проверка"].copy()

assert set(обучение["группа"]).isdisjoint(проверка["группа"])
assert set(обучение["ссылка"]).isdisjoint(проверка["ссылка"])
assert set(обучение["слова"]).isdisjoint(проверка["слова"])
print("Обучение:", len(обучение), "Проверка:", len(проверка))

Обучение: 172 Проверка: 76


## 4. Обучаем модель

`TfidfVectorizer` считает, насколько слова характерны для статьи
`min_df=2` оставляет слова, встречающиеся хотя бы в двух обучающих публикациях.
`max_df=0.98` убирает слова почти из всех статей.
`sublinear_tf=True` смягчает влияние многократного повторения одного слова.
`strip_accents="unicode"` одинаково обрабатывает буквы с диакритикой.

`class_weight="balanced"` учитывает, что слабых сигналов больше, чем зрелых технологий
`max_iter=2000` — предел работы алгоритма

In [ ]:
модель = make_pipeline(
    TfidfVectorizer(min_df=2, max_df=0.98, sublinear_tf=True, strip_accents="unicode"),
    LogisticRegression(class_weight="balanced", max_iter=2000)
)
модель.fit(обучение["слова"], обучение["класс"])
print("Модель обучена.")

Модель обучена.


## 5. Проверяем все три класса

`predict` возвращает выбранный класс. `predict_proba` даёт внутренние оценки классов.
Оценка «слабого сигнала» нужна для сортировки.

В таблице строки — исходные классы, столбцы — ответы модели.
Правильные ответы находятся на диагонали. Одна строка здесь — одна публикация.

In [ ]:
результат = проверка.drop(columns=["слова"]).copy()
результат["ответ_модели"] = модель.predict(проверка["слова"])
номер_класса = list(модель.classes_).index("слабый сигнал")
результат["оценка_сигнала"] = модель.predict_proba(проверка["слова"])[:, номер_класса]
правильно = (результат["класс"] == результат["ответ_модели"]).sum()
print(f"Правильно: {правильно} из {len(результат)} = {правильно / len(результат):.1%}")
результат.to_csv("Результаты/Все_ответы.csv", index=False, encoding="utf-8-sig")
pd.crosstab(результат["класс"], результат["ответ_модели"])

Правильно: 72 из 76 = 94.7%


ответ_модели,зрелая технология,слабый сигнал,шум
класс,,,
зрелая технология,7,0,1
слабый сигнал,1,50,0
шум,0,2,15


## 6. Собираем топ-15 и сверяем с Excel

Берём ответы «слабый сигнал», сортируем по оценке, оставляем одну публикацию
на группу технологии и первые 15 групп.

In [ ]:
топ15 = (результат[результат["ответ_модели"] == "слабый сигнал"]
    .sort_values("оценка_сигнала", ascending=False)
    .drop_duplicates("группа")
    .head(15).copy())
совпадений = топ15["номера_в_excel"].ne("").sum()
print(f"Совпадений с Excel: {совпадений} из 15 = {совпадений / 15:.1%}")
топ15.to_csv("Результаты/Топ15.csv", index=False, encoding="utf-8-sig")
топ15[["номера_в_excel", "технология", "оценка_сигнала", "ссылка"]].reset_index(drop=True)

Совпадений с Excel: 15 из 15 = 100.0%


,номера_в_excel,технология,оценка_сигнала,ссылка
0,81,Самообучающиеся контуры управления: RL прямо н...,0.697492,https://www.nature.com/articles/s41598-025-289...
1,75,Маршрутизирующий слой «устройство ↔ облако» ка...,0.696406,https://zylos.ai/en/research/2026-05-10-local-...
2,37,Роевая робототехника выходит из лаборатории (с...,0.663177,https://neuronfeed.com/startups/unbox-robotics
3,64,Эгоцентрическое человеческое видео как «дешёвы...,0.660142,https://arxiv.org/html/2606.17200v1
4,77,Сканирование бэкдоров в open-weight LLM «в мас...,0.645568,https://www.microsoft.com/en-us/security/blog/...
5,74,Локальные SLM-рантаймы как «анти-Big-Tech» аль...,0.633383,https://techfundingnews.com/nobodywho-2m-pre-s...
6,97,Биогибридные приводы на живой мышечной ткани,0.632379,https://www.nature.com/articles/s44182-025-000...
7,48,Machine unlearning как средство ремедиации рис...,0.626973,https://www.unite.ai/hirundo-raises-8m-to-tack...
8,23,Дифференциально-приватные синтетические данные...,0.620208,https://seedlist.com/startups/secludy.html
9,5,CAD-native «engineering intelligence»: ИИ-сурр...,0.619936,https://develop3d.com/cad/infinitform-sees-see...


## 7. Сохраняем проверенную модель

Сохраняется **та же самая модель**, которую мы только что проверили.
Она обучена на 172 публикациях; контрольные 76 в её обучение не добавляются.

In [ ]:
with open("Модель.pkl", "wb") as файл:
    pickle.dump(модель, файл)
print("Готово: Модель.pkl")

Готово: Модель.pkl


## 8. Применяем сохранённую модель

Для нового материала подайте **очищенный английский текст статьи**

Чтобы оценить свой материал, замените значение
`текст` обычным текстом английской статьи. Пустой текст, один заголовок и русская
статья не входят в подтверждённую область применения.

In [ ]:
with open("Модель.pkl", "rb") as файл:
    готовая_модель = pickle.load(файл)

текст = проверка.iloc[0]["слова"]
print("Класс:", готовая_модель.predict([текст])[0])
pd.Series(готовая_модель.predict_proba([текст])[0], index=готовая_модель.classes_, name="Оценка")

Класс: слабый сигнал


зрелая технология    0.203184
слабый сигнал        0.596641
шум                  0.200175
Name: Оценка, dtype: float64

## 9. Проверяем гипотезу о баллах Excel

In [ ]:
сигналы = pd.read_csv("Данные/100_сигналов.csv")
сигналы["расчёт"] = сигналы["балл_стадии"] + сигналы["балл_тренда"]
print("Формула совпала в", (сигналы["расчёт"] == сигналы["балл_excel"]).sum(), "из 100 строк")
сигналы.loc[сигналы["расчёт"] != сигналы["балл_excel"],
    ["номер", "технология", "балл_excel", "балл_стадии", "балл_тренда", "расчёт"]]

Формула совпала в 97 из 100 строк


,номер,технология,балл_excel,балл_стадии,балл_тренда,расчёт
65,66,Браузерно-нативные автономные ИИ-агенты компла...,5,4,3,7
68,69,Протоколы авторизации платежей ИИ-агентов (x40...,5,4,3,7
92,93,Депозитные токены банков на публичных блокчейнах,4,4,2,6
